In [ ]:
import pandas as pd
import requests as r
import getpass, pprint, time, os, json
import geopandas as gpd

In [ ]:
#Set working Dir
inDir = '/Users/tillweiss/Desktop/MODSNOW'         
os.chdir(inDir)    
inDir

In [ ]:
user = 'weiss14'     
password = "bemcog-bixqe3-bykpaZ"
api = 'https://appeears.earthdatacloud.nasa.gov/api/'  

In [ ]:
token_response = r.post('{}login'.format(api), auth=(user, password)).json() 
token = token_response['token']                      
head = {'Authorization': 'Bearer {}'.format(token)}

In [ ]:
#Optional find prods with NDVI
product_response = r.get('{}product'.format(api)).json()                        
print('AρρEEARS currently supports {} products.'.format(len(product_response)))

products = {p['ProductAndVersion']: p for p in product_response} 
prodNames = {p['ProductAndVersion'] for p in product_response} # Make list of all products (including version)
for p in prodNames:                                            # Make for loop to search list of products 'Description' for a keyword                
    if 'NDVI' in products[p]['Description']:
        pprint.pprint(products[p])

In [ ]:
#List alyers of prodct
prods = ['MOD13Q1.061']   
lst_response = r.get('{}product/{}'.format(api, prods[0])).json()  # Request layers for the 2nd product (index 1) in the list: MOD11A2.061
#list(lst_response.keys())

In [ ]:
#Set AOI

nps = gpd.read_file('/Users/tillweiss/Desktop/MODSNOW/data/Khorezm_region/khorezm_irrigated/khorezm_irrigated.shp')
print(nps.head()) 

In [ ]:
nps_gc = nps[nps['id']==0]  
nps_gc = nps_gc[['geometry']].to_json() 
nps_gc = json.loads(nps_gc) 

projections = r.get('{}spatial/proj'.format(api)).json()  
projs = {}                                 
for p in projections: projs[p['Name']] = p 
list(projs.keys())  

In [ ]:
#Set task configurations
task_name = 'MODSNOW_irrigated_MarJul_2000_2024'
task_type = 'area'
proj = projs['geographic']['Name']
outFormat = ['geotiff', 'netcdf4']
recurring = False

# Use a single continuous date range (API-safe)
startDate = "03-01-2000"
endDate   = "08-31-2025"

# ----------------------------
# Task definition
# ----------------------------
task = {
    'task_type': task_type,
    'task_name': task_name,
    'params': {
        'dates': [{
            'startDate': startDate,
            'endDate': endDate
        }],
        'layers': [
            {
                'product': 'MOD13Q1.061',
                'layer': '_250m_16_days_NDVI'
            }
        ],
        'qualityFilters': {
            'MOD13Q1.061': {
                'SummaryQA': [0]
            }
        },
        'output': {
            'format': {'type': outFormat[1]},  # netcdf4
            'projection': proj
        },
        'geo': nps_gc
    }
}


# ----------------------------
# Submit task
# ----------------------------
task_response = r.post(f'{api}task', json=task, headers=head).json()

if 'task_id' not in task_response:
    raise RuntimeError(f"Task submission failed: {task_response}")

task_id = task_response['task_id']
print(task_id)

# ----------------------------
# Monitor task
# ----------------------------
#starttime = time.time()
#while r.get(f'{api}task/{task_id}', headers=head).json()['status'] != 'done':
#    status = r.get(f'{api}task/{task_id}', headers=head).json()['status']
#    print(status)
#    time.sleep(20.0 - ((time.time() - starttime) % 20.0))

print('done')

In [ ]:
#Monitor task
task_id = 'c52aa472-76d2-48e2-9464-002abb75c54e'

status = r.get(f'{api}task/{task_id}', headers=head).json()['status']
print(f"Current task status: {status}")

In [ ]:
#Download task
task_name = 'MODSNOW_irrigated_MarJul_2000_2024'
destDir = os.path.join(inDir, task_name)
os.makedirs(destDir, exist_ok=True)

bundle = r.get(f'{api}bundle/{task_id}', headers=head).json()

files = {f['file_id']: f['file_name'] for f in bundle['files']}

for file_id, fname in files.items():
    dl = r.get(
        f'{api}bundle/{task_id}/{file_id}',
        headers=head,
        stream=True,
        allow_redirects=True
    )

    filename = fname.split('/')[-1]
    filepath = os.path.join(destDir, filename)

    with open(filepath, 'wb') as out:
        for chunk in dl.iter_content(chunk_size=8192):
            out.write(chunk)

print(f"Download complete. Files are in:\n{destDir}")